In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import scipy.signal as signal
import scipy.stats as stats
import torch
import torch.nn as nn
import torch.optim as optim
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from funciones.cargador import obtener_una_muestra, FREQ_MUESTREO
import glob
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

In [ ]:
# def visualizar_espectrograma_interactivo(tensor, nombre_prueba):
#     res = detector_tiempo_frecuencia(tensor)
#     rafagas_orig, t_sec, f, Pxx_dB, log_energia_t, umbral_t, fft_global = res
    
#     t_ms = t_sec * 1000
#     f_mhz = f / 1e6
    
#     iq = tensor.cpu().numpy() if isinstance(tensor, torch.Tensor) else tensor
#     mag = np.abs(iq[0,:] + 1j*iq[1,:])
#     t_raw = (np.arange(len(mag))/FREQ_MUESTREO)*1000
    
#     # IMPORTANTE: 3 filas y 2 columnas puras para mantener la alineación
#     fig = make_subplots(
#         rows=3, cols=2,
#         column_widths=[0.85, 0.15],
#         row_heights=[0.25, 0.5, 0.25],
#         horizontal_spacing=0.04, # Aumentado ligeramente para las etiquetas del eje Y
#         vertical_spacing=0.1,
#         shared_xaxes=False, 
#         subplot_titles=(
#             "Magnitud Temporal (Lineal)", "", 
#             "Espectrograma STFT (dB)", "FFT Global (dB)", 
#             "Proyección de Energía (dB)", ""
#         )
#     )
    
#     # 1. Tiempo (Fila 1, Col 1)
#     fig.add_trace(go.Scatter(x=t_raw[::100], y=mag[::100], name='IQ',
#                              line=dict(color='rgba(200,200,200,0.5)', width=1)), row=1, col=1)
    
#     # 2. Espectrograma (Fila 2, Col 1)
#     fig.add_trace(go.Heatmap(z=Pxx_dB[::4, ::4], x=t_ms[::4], y=f_mhz[::4], 
#                              colorscale="viridis" ,showscale=False), row=2, col=1)
    
#     # 3. FFT (Fila 2, Col 2)
#     fig.add_trace(go.Scatter(x=fft_global, y=f_mhz, name='FFT',
#                              line=dict(color='yellow', width=2)), row=2, col=2)
    
#     # 4. Energía (Fila 3, Col 1)
#     fig.add_trace(go.Scatter(x=t_ms, y=log_energia_t, name='Energía',
#                              line=dict(color='cyan', width=2)), row=3, col=1)
#     fig.add_trace(go.Scatter(x=t_ms, y=np.full_like(t_ms, umbral_t), name='Umbral',
#                              line=dict(color='red', dash='dash')), row=3, col=1)
    
#     # --- VINCULACIÓN TOTAL Y ETIQUETAS DE EJES ---
#     fig.update_layout(
#         xaxis=dict(matches='x3', title="Tiempo (ms)"),
#         xaxis3=dict(title="Tiempo (ms)"),
#         xaxis5=dict(matches='x3', title="Tiempo (ms)"),
#         xaxis4=dict(title="Potencia (dB)"),
        
#         yaxis=dict(title="Amplitud"),
#         yaxis3=dict(title="Frecuencia (MHz)"),
#         yaxis4=dict(matches='y3'),
#         yaxis5=dict(title="Energía (dB)"),
        
#         template="plotly_dark",
#         height=950, width=1200,
#         showlegend=False,
#         title_text=f"Análisis Multidominio: {nombre_prueba}"
#     )

#     # Cajas rojas sincronizadas
#     for r in rafagas_orig:
#         t0, t1 = (r[0]/FREQ_MUESTREO)*1000, (r[1]/FREQ_MUESTREO)*1000
#         for i in [1, 2, 3]:
#             fig.add_vrect(x0=t0, x1=t1, fillcolor="red", opacity=0.1, line_width=0, row=i, col=1)

#     fig.show()

# Detector tiempo - frecuencia

In [ ]:
# def detector_tiempo_frecuencia(iq_tensor, fs=FREQ_MUESTREO, nperseg=1024, alpha=4.0, min_burst_ms=0.1, max_gap_ms=0.5):
#     if isinstance(iq_tensor, torch.Tensor):
#         iq_array = iq_tensor.cpu().numpy()
#     else:
#         iq_array = iq_tensor
        
#     x = iq_array[0, :] + 1j * iq_array[1, :]
    
#     f, t_sec, Zxx = signal.stft(x, fs=fs, window='hann', nperseg=nperseg, noverlap=nperseg//2, return_onesided=False)
    
#     f = np.fft.fftshift(f)
#     Zxx = np.fft.fftshift(Zxx, axes=0)
    
#     Pxx_dB = 10 * np.log10(np.abs(Zxx)**2 + 1e-12)
    
#     # Cálculo del Suelo de Ruido Real (Estadística 2D)
#     mediana_2d = np.median(Pxx_dB)
#     mad_2d = np.median(np.abs(Pxx_dB - mediana_2d))
    
#     sigma_est = mad_2d * 1.4826
#     umbral_val = mediana_2d + alpha * sigma_est

#     detecciones_temporales = np.any(Pxx_dB > umbral_val, axis=0)
    
#     cambios = np.diff(detecciones_temporales.astype(int))
#     inicios = np.where(cambios == 1)[0]
#     fines = np.where(cambios == -1)[0]
    
#     if detecciones_temporales[0]: inicios = np.insert(inicios, 0, 0)
#     if detecciones_temporales[-1]: fines = np.append(fines, len(t_sec) - 1)
        
#     dt = t_sec[1] - t_sec[0]
#     min_muestras_t = int((min_burst_ms / 1000.0) / dt)

#     rafagas_preliminares = []
#     for i in range(len(inicios)):
#         if (fines[i] - inicios[i]) >= min_muestras_t:
#             rafagas_preliminares.append((inicios[i], fines[i]))

#     max_gap_muestras = int((max_gap_ms / 1000.0) / dt)
#     rafagas_fusionadas = []
    
#     if rafagas_preliminares:
#         rafagas_preliminares.sort(key=lambda x: x[0])
#         inicio_actual, fin_actual = rafagas_preliminares[0]
        
#         for inicio_sig, fin_sig in rafagas_preliminares[1:]:
#             if inicio_sig <= (fin_actual + max_gap_muestras):
#                 fin_actual = max(fin_actual, fin_sig)
#             else:
#                 rafagas_fusionadas.append((inicio_actual, fin_actual))
#                 inicio_actual, fin_actual = inicio_sig, fin_sig
                
#         rafagas_fusionadas.append((inicio_actual, fin_actual))

#     rafagas_originales = []
#     hop_size = nperseg // 2
#     for inicio_stft, fin_stft in rafagas_fusionadas:
#         inicio_real = inicio_stft * hop_size
#         fin_real = min((fin_stft * hop_size) + nperseg, iq_array.shape[1])
#         rafagas_originales.append((inicio_real, fin_real))
        
#     max_energia_temporal = np.max(Pxx_dB, axis=0) 
            
#     return rafagas_originales, t_sec, f, Pxx_dB, max_energia_temporal, umbral_val

In [ ]:
def detector_tiempo_frecuencia(iq_tensor, fs=FREQ_MUESTREO, nperseg=1024, alpha=3.5, min_burst_ms=0.1, max_gap_ms=0.5):
    if isinstance(iq_tensor, torch.Tensor):
        iq_array = iq_tensor.cpu().numpy()
    else:
        iq_array = iq_tensor
        
    x = iq_array[0, :] + 1j * iq_array[1, :]
    
    f, t_sec, Zxx = signal.stft(x, fs=fs, window='hann', nperseg=nperseg, noverlap=nperseg//2, return_onesided=False)
    f = np.fft.fftshift(f)
    Zxx = np.fft.fftshift(Zxx, axes=0)
    Pxx = np.abs(Zxx)**2 
    Pxx_dB = 10 * np.log10(Pxx + 1e-15)
    
    # 1. PROYECCIÓN TEMPORAL (Suma por columnas - Lo que pidió el tutor)
    # Detecta ráfagas de banda ancha o espectro ensanchado integrando potencia
    energia_t = np.max(Pxx, axis=0) 
    log_energia_t = 10 * np.log10(energia_t + 1e-15)
    
    # 2. PROYECCIÓN FRECUENCIAL (Suma por filas - FFT Promediada)
    fft_global = 10 * np.log10(np.mean(Pxx, axis=1) + 1e-15)
    
    # 3. ESTADÍSTICA ROBUSTA (Sobre la proyección temporal)
    mediana_t = np.median(log_energia_t)
    mad_t = np.median(np.abs(log_energia_t - mediana_t))
    umbral_t = mediana_t + alpha * (mad_t * 1.4826)
    
    # 4. DETECCIÓN
    detecciones = log_energia_t > umbral_t
    
    cambios = np.diff(detecciones.astype(int))
    inicios = np.where(cambios == 1)[0]
    fines = np.where(cambios == -1)[0]
    
    if detecciones[0]: inicios = np.insert(inicios, 0, 0)
    if detecciones[-1]: fines = np.append(fines, len(t_sec) - 1)
        
    dt = t_sec[1] - t_sec[0]
    min_muestras_t = int((min_burst_ms / 1000.0) / dt)
    
    # Fusión de ráfagas
    max_gap_muestras = int((max_gap_ms / 1000.0) / dt)
    rafagas_pre = []
    for i in range(len(inicios)):
        if (fines[i] - inicios[i]) >= min_muestras_t:
            rafagas_pre.append((inicios[i], fines[i]))
            
    rafagas_fusionadas = []
    if rafagas_pre:
        rafagas_pre.sort(key=lambda x: x[0])
        curr_in, curr_fi = rafagas_pre[0]
        for nxt_in, nxt_fi in rafagas_pre[1:]:
            if nxt_in <= (curr_fi + max_gap_muestras):
                curr_fi = max(curr_fi, nxt_fi)
            else:
                rafagas_fusionadas.append((curr_in, curr_fi))
                curr_in, curr_fi = nxt_in, nxt_fi
        rafagas_fusionadas.append((curr_in, curr_fi))

    # Índices originales para recorte
    rafagas_orig = []
    hop = nperseg // 2
    for s_stft, e_stft in rafagas_fusionadas:
        rafagas_orig.append((s_stft * hop, min((e_stft * hop) + nperseg, iq_array.shape[1])))
            
    return rafagas_orig, t_sec, f, Pxx_dB, log_energia_t, umbral_t, fft_global

In [ ]:
for index in range(0,10):
    tensor_taranis, _, _, _ = obtener_una_muestra(target=2, snr=-2, index=index)
    visualizar_espectrograma_interactivo(tensor_taranis, "Dron")

# tensor_dji, _, _, _ = obtener_una_muestra(target=1, snr=0)
# visualizar_espectrograma_interactivo(tensor_dji, "Dron")

# tensor_ruido, _, _, _ = obtener_una_muestra(target=4, snr=0)
# visualizar_espectrograma_interactivo(tensor_ruido, "Ruido: Interferencia Wi-Fi/BT")

# Iterative Excision Sigma-clipping

In [ ]:
def detector_tiempo_frecuencia(iq_tensor, fs=FREQ_MUESTREO, nperseg=1024, pfa=1e-3, min_burst_ms=0.1, max_gap_ms=0.5):
    """
    Detector SIGINT ciego mediante Escisión Iterativa (Iterative Sigma-Clipping).
    Totalmente agnóstico al duty-cycle de la señal o al ancho de banda.
    """
    if isinstance(iq_tensor, torch.Tensor):
        iq_array = iq_tensor.cpu().numpy()
    else:
        iq_array = iq_tensor
        
    x = iq_array[0, :] + 1j * iq_array[1, :]
    
    # 1. Transformación STFT
    f, t_sec, Zxx = signal.stft(x, fs=fs, window='hann', nperseg=nperseg, noverlap=nperseg//2, return_onesided=False)
    f = np.fft.fftshift(f)
    Zxx = np.fft.fftshift(Zxx, axes=0)
    Pxx = np.abs(Zxx)**2 
    
    # Proyección Híbrida: Máximo para no diluir banda estrecha
    energia_t = np.max(Pxx, axis=0)
    log_energia_t = 10 * np.log10(energia_t + 1e-15)
    
    # ====================================================================
    # 2. SEPARACIÓN CIEGA DE DISTRIBUCIONES (ITERATIVE EXCISION)
    # ====================================================================
    ruido_estimado = log_energia_t.copy()
    
    # Bucle iterativo de "pelado" de la señal (máximo 15 iteraciones para garantizar real-time)
    for _ in range(15):
        mu = np.mean(ruido_estimado)
        sigma = np.std(ruido_estimado)
        
        # El 3-sigma elimina el 99.7% de los outliers en una distribución normal
        umbral_corte = mu + 3 * sigma
        
        # Filtramos: nos quedamos solo con lo que se parece al ruido
        ruido_nuevo = ruido_estimado[ruido_estimado < umbral_corte]
        
        # Condición de convergencia: si no hemos eliminado ningún punto nuevo, hemos aislado el ruido
        if len(ruido_nuevo) == len(ruido_estimado) or len(ruido_nuevo) < 2:
            break
            
        ruido_estimado = ruido_nuevo
        
    # ====================================================================
    # 3. CÁLCULO DE UMBRAL ESTRICTO BASADO EN PFA (Probability of False Alarm)
    # ====================================================================
    mu_ruido_puro = np.mean(ruido_estimado)
    sigma_ruido_puro = np.std(ruido_estimado)
    
    # Calculamos el multiplicador Z para la PFA deseada (ej. 1e-3 -> Z ~= 3.09)
    factor_z = stats.norm.ppf(1 - pfa)
    umbral_t = mu_ruido_puro + factor_z * sigma_ruido_puro
    
    # 4. DETECCIÓN
    detecciones = log_energia_t > umbral_t
    
    # 5. EXTRACCIÓN Y FUSIÓN (Idéntico a las versiones anteriores)
    cambios = np.diff(detecciones.astype(int))
    inicios = np.where(cambios == 1)[0]
    fines = np.where(cambios == -1)[0]
    
    if detecciones[0]: inicios = np.insert(inicios, 0, 0)
    if detecciones[-1]: fines = np.append(fines, len(t_sec) - 1)
        
    dt = t_sec[1] - t_sec[0]
    min_muestras_t = int((min_burst_ms / 1000.0) / dt)
    max_gap_muestras = int((max_gap_ms / 1000.0) / dt)
    
    rafagas_pre = []
    for i in range(len(inicios)):
        if (fines[i] - inicios[i]) >= min_muestras_t:
            rafagas_pre.append((inicios[i], fines[i]))
            
    rafagas_fusionadas = []
    if rafagas_pre:
        rafagas_pre.sort(key=lambda x: x[0])
        curr_in, curr_fi = rafagas_pre[0]
        for nxt_in, nxt_fi in rafagas_pre[1:]:
            if nxt_in <= (curr_fi + max_gap_muestras):
                curr_fi = max(curr_fi, nxt_fi)
            else:
                rafagas_fusionadas.append((curr_in, curr_fi))
                curr_in, curr_fi = nxt_in, nxt_fi
        rafagas_fusionadas.append((curr_in, curr_fi))

    rafagas_orig = []
    hop = nperseg // 2
    for s_stft, e_stft in rafagas_fusionadas:
        rafagas_orig.append((s_stft * hop, min((e_stft * hop) + nperseg, iq_array.shape[1])))
        
    fft_global = 10 * np.log10(np.mean(Pxx, axis=1) + 1e-15)
            
    return rafagas_orig, t_sec, f, 10 * np.log10(Pxx + 1e-15), log_energia_t, umbral_t, fft_global

In [ ]:
def visualizar_espectrograma_interactivo(tensor, nombre_prueba):
    res = detector_tiempo_frecuencia(tensor)
    rafagas_orig, t_sec, f, Pxx_dB, log_energia_t, umbral_t, fft_global = res
    
    t_ms = t_sec * 1000
    f_mhz = f / 1e6
    
    iq = tensor.cpu().numpy() if isinstance(tensor, torch.Tensor) else tensor
    mag = np.abs(iq[0,:] + 1j*iq[1,:])
    t_raw = (np.arange(len(mag))/FREQ_MUESTREO)*1000
    
    # EL CÓDIGO QUE TÚ ME PASASTE Y QUE FUNCIONABA PERFECTO
    fig = make_subplots(
        rows=3, cols=2,
        column_widths=[0.85, 0.15],
        row_heights=[0.25, 0.5, 0.25],
        horizontal_spacing=0.02,
        vertical_spacing=0.075,
        shared_xaxes=False, 
        subplot_titles=("Magnitud Temporal", "", "Espectrograma STFT", "FFT", "Energía", "")
    )
    
    # 1. Tiempo (Fila 1, Col 1)
    fig.add_trace(go.Scatter(x=t_raw[::100], y=mag[::100], name='IQ',
                             line=dict(color='rgba(200,200,200,0.5)', width=1)), row=1, col=1)
    
    # 2. Espectrograma (Fila 2, Col 1) -> EL EJE MAESTRO
    fig.add_trace(go.Heatmap(z=Pxx_dB[::4, ::4], x=t_ms[::4], y=f_mhz[::4], 
                             colorscale='Viridis', showscale=False), row=2, col=1)
    
    # 3. FFT (Fila 2, Col 2) -> Comparte Y con Espectrograma
    fig.add_trace(go.Scatter(x=fft_global, y=f_mhz, name='FFT',
                             line=dict(color='yellow', width=2)), row=2, col=2)
    
    # 4. Energía (Fila 3, Col 1)
    fig.add_trace(go.Scatter(x=t_ms, y=log_energia_t, name='Energía',
                             line=dict(color='cyan', width=2)), row=3, col=1)
    fig.add_trace(go.Scatter(x=t_ms, y=np.full_like(t_ms, umbral_t), name='Umbral',
                             line=dict(color='red', dash='dash')), row=3, col=1)
    
    # --- VINCULACIÓN TOTAL DE EJES (Tu versión original) ---
    fig.update_layout(
        xaxis=dict(matches='x3'),  # Fila 1 vinculada a Espectrograma
        xaxis5=dict(matches='x3'), # Fila 3 vinculada a Espectrograma
        yaxis4=dict(matches='y3'), # FFT vinculada a Y de Espectrograma
        template="plotly_dark",
        height=900, width=1200,
        showlegend=False,
        title_text=nombre_prueba
    )
    
    # --- AÑADIR UNIDADES DE FORMA SEGURA (Sin romper el layout) ---
    fig.update_xaxes(title_text="Tiempo (ms)", row=3, col=1)
    fig.update_xaxes(title_text="Potencia (dB)", row=2, col=2)
    
    fig.update_yaxes(title_text="Amplitud", row=1, col=1)
    fig.update_yaxes(title_text="Frecuencia (MHz)", row=2, col=1)
    fig.update_yaxes(title_text="Energía (dB)", row=3, col=1)

    # Cajas rojas sincronizadas
    for r in rafagas_orig:
        t0, t1 = (r[0]/FREQ_MUESTREO)*1000, (r[1]/FREQ_MUESTREO)*1000
        for i in [1, 2, 3]:
            fig.add_vrect(x0=t0, x1=t1, fillcolor="red", opacity=0.1, line_width=0, row=i, col=1)

    fig.show()

In [ ]:
for index in range(10,20):
    tensor_taranis, _, _, _ = obtener_una_muestra(target=2, snr=6, index=index)
    visualizar_espectrograma_interactivo(tensor_taranis, "Dron")

# tensor_dji, _, _, _ = obtener_una_muestra(target=1, snr=0)
# visualizar_espectrograma_interactivo(tensor_dji, "Dron")

# tensor_ruido, _, _, _ = obtener_una_muestra(target=4, snr=0)
# visualizar_espectrograma_interactivo(tensor_ruido, "Ruido: Interferencia Wi-Fi/BT")

# Estimación por Simetría de Cola Izquierda (Half-Sample Mode)

In [ ]:
def detector_tiempo_frecuencia(iq_tensor, fs=FREQ_MUESTREO, nperseg=1024, pfa=1e-10, min_burst_ms=0.08, max_gap_ms=0.1):
    """
    Detector SIGINT Teórico. 
    Usa la Moda para el suelo de ruido y la ley exponencial para el PFA.
    """
    if isinstance(iq_tensor, torch.Tensor):
        iq_array = iq_tensor.cpu().numpy()
    else:
        iq_array = iq_tensor
        
    x = iq_array[0, :] + 1j * iq_array[1, :]
    
    # 1. Transformación STFT
    f, t_sec, Zxx = signal.stft(x, fs=fs, window='hann', nperseg=nperseg, noverlap=nperseg//2, return_onesided=False)
    f = np.fft.fftshift(f)
    Zxx = np.fft.fftshift(Zxx, axes=0)
    Pxx = np.abs(Zxx)**2 
    
    # Proyección Máxima (Mantiene la forma real de la señal, sin diluir)
    energia_t = np.max(Pxx, axis=0)
    log_energia_t = 10 * np.log10(energia_t + 1e-15)
    
    # ====================================================================
    # 2. ESTIMACIÓN DEL SUELO DE RUIDO (Inmune a la densidad de la señal)
    # ====================================================================
    # Calculamos el histograma para encontrar la Moda (el valle más común).
    # Aunque el dron ocupe el 90% del tiempo, el 10% de ruido forma el pico más denso.
    hist, bin_edges = np.histogram(log_energia_t, bins=100)
    idx_peak = np.argmax(hist)
    ruido_db = (bin_edges[idx_peak] + bin_edges[idx_peak+1]) / 2.0
    
    # ====================================================================
    # 3. CÁLCULO DE UMBRAL TEÓRICO (Ley Exponencial de Potencia)
    # ====================================================================
    # Para ruido térmico (AWGN), la diferencia entre el suelo de ruido y el 
    # umbral viene dictada puramente por la PFA deseada. 
    # Para PFA = 1e-6, este offset es exactamente ~11.4 dB.
    offset_db = 10 * np.log10(-np.log(pfa))
    umbral_t = ruido_db + offset_db
    
    # 4. DETECCIÓN (Sobre la señal cruda y real)
    detecciones = log_energia_t > umbral_t
    
    # 5. EXTRACCIÓN Y FUSIÓN MORFOLÓGICA
    cambios = np.diff(detecciones.astype(int))
    inicios = np.where(cambios == 1)[0]
    fines = np.where(cambios == -1)[0]
    
    if detecciones[0]: inicios = np.insert(inicios, 0, 0)
    if detecciones[-1]: fines = np.append(fines, len(t_sec) - 1)
        
    dt = t_sec[1] - t_sec[0]
    min_muestras_t = int((min_burst_ms / 1000.0) / dt)
    max_gap_muestras = int((max_gap_ms / 1000.0) / dt)
    
    rafagas_pre = []
    for i in range(len(inicios)):
        if (fines[i] - inicios[i]) >= min_muestras_t:
            rafagas_pre.append((inicios[i], fines[i]))
            
    rafagas_fusionadas = []
    if rafagas_pre:
        rafagas_pre.sort(key=lambda x: x[0])
        curr_in, curr_fi = rafagas_pre[0]
        for nxt_in, nxt_fi in rafagas_pre[1:]:
            if nxt_in <= (curr_fi + max_gap_muestras):
                curr_fi = max(curr_fi, nxt_fi)
            else:
                rafagas_fusionadas.append((curr_in, curr_fi))
                curr_in, curr_fi = nxt_in, nxt_fi
        rafagas_fusionadas.append((curr_in, curr_fi))

    rafagas_orig = []
    hop = nperseg // 2
    for s_stft, e_stft in rafagas_fusionadas:
        rafagas_orig.append((s_stft * hop, min((e_stft * hop) + nperseg, iq_array.shape[1])))
        
    fft_global = 10 * np.log10(np.mean(Pxx, axis=1) + 1e-15)
            
    return rafagas_orig, t_sec, f, 10 * np.log10(Pxx + 1e-15), log_energia_t, umbral_t, fft_global

In [ ]:
def visualizar_espectrograma_interactivo(tensor, nombre_prueba):
    res = detector_tiempo_frecuencia(tensor)
    rafagas_orig, t_sec, f, Pxx_dB, log_energia_t, umbral_t, fft_global = res
    
    t_ms = t_sec * 1000
    f_mhz = f / 1e6
    
    iq = tensor.cpu().numpy() if isinstance(tensor, torch.Tensor) else tensor
    mag = np.abs(iq[0,:] + 1j*iq[1,:])
    t_raw = (np.arange(len(mag))/FREQ_MUESTREO)*1000
    
    # EL CÓDIGO QUE TÚ ME PASASTE Y QUE FUNCIONABA PERFECTO
    fig = make_subplots(
        rows=3, cols=2,
        column_widths=[0.85, 0.15],
        row_heights=[0.25, 0.5, 0.25],
        horizontal_spacing=0.02,
        vertical_spacing=0.075,
        shared_xaxes=False, 
        subplot_titles=("Magnitud Temporal", "", "Espectrograma STFT", "FFT", "Energía", "")
    )
    
    # 1. Tiempo (Fila 1, Col 1)
    fig.add_trace(go.Scatter(x=t_raw[::100], y=mag[::100], name='IQ',
                             line=dict(color='rgba(200,200,200,0.5)', width=1)), row=1, col=1)
    
    # 2. Espectrograma (Fila 2, Col 1) -> EL EJE MAESTRO
    fig.add_trace(go.Heatmap(z=Pxx_dB[::4, ::4], x=t_ms[::4], y=f_mhz[::4], 
                             colorscale='Viridis', showscale=False), row=2, col=1)
    
    # 3. FFT (Fila 2, Col 2) -> Comparte Y con Espectrograma
    fig.add_trace(go.Scatter(x=fft_global, y=f_mhz, name='FFT',
                             line=dict(color='yellow', width=2)), row=2, col=2)
    
    # 4. Energía (Fila 3, Col 1)
    fig.add_trace(go.Scatter(x=t_ms, y=log_energia_t, name='Energía',
                             line=dict(color='cyan', width=2)), row=3, col=1)
    fig.add_trace(go.Scatter(x=t_ms, y=np.full_like(t_ms, umbral_t), name='Umbral',
                             line=dict(color='red', dash='dash')), row=3, col=1)
    
    # --- VINCULACIÓN TOTAL DE EJES (Tu versión original) ---
    fig.update_layout(
        xaxis=dict(matches='x3'),  # Fila 1 vinculada a Espectrograma
        xaxis5=dict(matches='x3'), # Fila 3 vinculada a Espectrograma
        yaxis4=dict(matches='y3'), # FFT vinculada a Y de Espectrograma
        template="plotly_dark",
        height=900, width=1200,
        showlegend=False,
        title_text=nombre_prueba
    )
    
    # --- AÑADIR UNIDADES DE FORMA SEGURA (Sin romper el layout) ---
    fig.update_xaxes(title_text="Tiempo (ms)", row=3, col=1)
    fig.update_xaxes(title_text="Potencia (dB)", row=2, col=2)
    
    fig.update_yaxes(title_text="Amplitud", row=1, col=1)
    fig.update_yaxes(title_text="Frecuencia (MHz)", row=2, col=1)
    fig.update_yaxes(title_text="Energía (dB)", row=3, col=1)

    # Cajas rojas sincronizadas
    for r in rafagas_orig:
        t0, t1 = (r[0]/FREQ_MUESTREO)*1000, (r[1]/FREQ_MUESTREO)*1000
        for i in [1, 2, 3]:
            fig.add_vrect(x0=t0, x1=t1, fillcolor="red", opacity=0.1, line_width=0, row=i, col=1)

    fig.show()

In [ ]:
for index in range(5,6):
    tensor_taranis, _, _, _ = obtener_una_muestra(target=4, snr=30, index=index)
    visualizar_espectrograma_interactivo(tensor_taranis, "Dron")

In [ ]:
# --- CONFIGURACIÓN ---
OUTPUT_DIR = "dataset_recortado_v1"
TARGET_MAP = {
    0: "DJI", 1: "FutabaT14", 2: "FutabaT7", 
    3: "Graupner", 4: "Noise", 5: "Taranis", 6: "Turnigy"
}
SNRS_A_PROCESAR = [-10]
SAMPLES_PER_CASE = 10 # Ajusta según necesites

def fabricar_y_analizar_dataset(visualizar_paso=2):
    """
    Genera el dataset, calcula estadísticas e inspecciona visualmente 
    muestras de forma diezmada.
    """
    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
    
    resultados_iteracion = []
    vis_count = 0 # Contador para la decimación visual
    
    print(f"🚀 Iniciando Fábrica de Datasets e Inspección Visual...")
    
    for target_id, name in TARGET_MAP.items():
        class_path = os.path.join(OUTPUT_DIR, name)
        if not os.path.exists(class_path): os.makedirs(class_path)
        
        for snr in SNRS_A_PROCESAR:
            num_detecciones = 0
            duraciones = []
            
            # Estrategia Hard Negative Mining para la clase Noise
            pfa_actual = 1e-2 if name == "Noise" else 1e-6
            
            for i in range(SAMPLES_PER_CASE):
                try:
                    # 1. Cargar señal
                    tensor, _, _, _ = obtener_una_muestra(target=target_id, snr=snr, index=i)
                    
                    # 2. Detectar (Moda + PFA)
                    res = detector_tiempo_frecuencia(tensor, fs=FREQ_MUESTREO, pfa=pfa_actual)
                    rafagas, t_sec = res[0], res[1]
                    
                    # --- INSPECCIÓN VISUAL INTEGRADA ---
                    # Visualizamos 1 de cada 'visualizar_paso' muestras cargadas
                    if vis_count % visualizar_paso == 0:
                        titulo = f"INSPECCIÓN: {name} | SNR: {snr}dB | Muestra: {i} (PFA: {pfa_actual})"
                        visualizar_espectrograma_interactivo(tensor, titulo)
                    
                    vis_count += 1
                    
                    # 3. Procesar ráfagas detectadas
                    for r_idx, (start, end) in enumerate(rafagas):
                        burst_iq = tensor[:, start:end]
                        
                        # Normalización Z-Score (Crucial para la Red Neuronal)
                        burst_iq = (burst_iq - burst_iq.mean()) / (burst_iq.std() + 1e-15)
                        
                        # Guardar
                        fname = f"T{target_id}_SNR{snr}_I{i}_B{r_idx}.npy"
                        # np.save(os.path.join(class_path, fname), burst_iq.cpu().numpy())
                        
                        num_detecciones += 1
                        # Calcular duración en ms
                        duraciones.append((end - start) * (t_sec[1] - t_sec[0]) * 1000)
                        
                except FileNotFoundError:
                    continue
            
            # Registro de estadísticas por bloque SNR/Clase
            resultados_iteracion.append({
                "Clase": name,
                "SNR (dB)": snr,
                "Total Detecciones": num_detecciones,
                "Duración Media (ms)": np.mean(duraciones) if duraciones else 0,
                "Estado": "OK" if num_detecciones > 0 else "Vacio"
            })

    return pd.DataFrame(resultados_iteracion)

In [ ]:
# --- EJECUCIÓN ---
# visualizar_paso=2 significa que verás 1 de cada 2 archivos procesados.
df_final = fabricar_y_analizar_dataset(visualizar_paso=2)

# Mostrar tabla final resumen
import IPython.display as display
display.display(df_final)

# LSTM Aproach

In [ ]:
# Configuración Global
OUTPUT_DIR = "dataset_recortado_v1"
TARGET_MAP = {0: "DJI", 1: "FutabaT14", 2: "FutabaT7", 3: "Graupner", 4: "Noise", 5: "Taranis", 6: "Turnigy"}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FACTOR_DECIMACION = 70 # 100 puntos = 0.5ms (Ajustado a tus ráfagas reales)

print(f"✅ Sistema listo en: {DEVICE}")

In [ ]:
def preprocesar_opcion_b(iq_data, factor=FACTOR_DECIMACION):
    if iq_data is None or iq_data.size == 0: return None
    # 1. Envolvente de magnitud
    mag = np.sqrt(iq_data[0]**2 + iq_data[1]**2)
    # 2. Longitud mínima requerida para 1 ventana de la LSTM (100 puntos)
    if len(mag) < (100 * factor): return None
    
    n_puntos = len(mag) // factor
    mag_comprimida = mag[:n_puntos * factor].reshape(-1, factor).mean(axis=1)
    
    # 3. Normalización Z-Score
    std = np.std(mag_comprimida)
    if std == 0: return None
    return (mag_comprimida - np.mean(mag_comprimida)) / (std + 1e-15)

In [ ]:
def fabricar_dataset_maestro(samples_per_case=100):
    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
    from scipy import signal
    print("🚀 Fabricando archivos maestros en disco...")
    
    for tid, name in TARGET_MAP.items():
        path = os.path.join(OUTPUT_DIR, name)
        if not os.path.exists(path): os.makedirs(path)
        snr_val = 30 if name != "Noise" else 0
        
        for i in range(samples_per_case):
            try:
                tensor, _, _, _ = obtener_una_muestra(target=tid, snr=snr_val, index=i)
                iq = tensor.cpu().numpy()
                
                if name == "Noise":
                    np.save(os.path.join(path, f"Noise_{i}.npy"), iq)
                else:
                    # Detector de umbral para guardar solo la ráfaga
                    mag_sq = iq[0]**2 + iq[1]**2
                    umbral = np.median(mag_sq) * 10 
                    det = mag_sq > umbral
                    cambios = np.diff(det.astype(int))
                    inicios, fines = np.where(cambios==1)[0], np.where(cambios==-1)[0]
                    
                    for r_idx, (s, e) in enumerate(zip(inicios, fines)):
                        # Guardamos ráfagas que tengan al menos 0.5ms (7000 muestras)
                        if (e - s) >= (100 * FACTOR_DECIMACION):
                            np.save(os.path.join(path, f"T{tid}_I{i}_B{s}.npy"), iq[:, s:e])
            except: break
    print(f"✅ Archivos maestros listos en '{OUTPUT_DIR}'")

fabricar_dataset_maestro()

In [ ]:
def generar_dataset_lacy_pytorch(n_muestras_total=20000, seq_len=100):
    X, y = [], []
    drones = ["DJI", "FutabaT14", "FutabaT7", "Graupner", "Taranis", "Turnigy"]
    
    # Precarga en RAM
    def cargar(nombres):
        cache = []
        for n in nombres:
            for f in glob.glob(f"{OUTPUT_DIR}/{n}/*.npy"):
                vec = preprocesar_opcion_b(np.load(f))
                if vec is not None and len(vec) >= seq_len: cache.append(vec)
        return cache

    print("📂 Cargando y comprimiendo señales...")
    data_d = cargar(drones)
    data_r = cargar(["Noise"])
    print(f"✔️ Drones: {len(data_d)} | Ruido: {len(data_r)}")

    # Generación de ventanas con ruido Lacy (-32 a -2 dB)
    for _ in tqdm(range(n_muestras_total // 2), desc="Generando muestras"):
        # Clase 1: Señal + Ruido
        vec_d = data_d[np.random.randint(len(data_d))]
        s = np.random.randint(0, len(vec_d) - seq_len)
        ventana = vec_d[s:s+seq_len].reshape(-1, 1)
        snr_db = np.random.uniform(-32, -2)
        p_noise = np.mean(ventana**2) / (10**(snr_db / 10))
        X.append(ventana + np.random.normal(0, np.sqrt(p_noise), ventana.shape)); y.append(1)
        
        # Clase 0: Solo Ruido
        vec_r = data_r[np.random.randint(len(data_r))]
        s = np.random.randint(0, len(vec_r) - seq_len)
        X.append(vec_r[s:s+seq_len].reshape(-1, 1)); y.append(0)

    X_train, X_test, y_train, y_test = train_test_split(np.array(X), np.array(y), test_size=0.2)
    train_ds = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
    test_ds = TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test))
    return DataLoader(train_ds, batch_size=1000, shuffle=True), DataLoader(test_ds, batch_size=1000)

train_loader, test_loader = generar_dataset_lacy_pytorch()

In [ ]:
epochs = 300

In [ ]:
class DetectorLacy(nn.Module):
    def __init__(self):
        super(DetectorLacy, self).__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=128, batch_first=True)
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(128, 64); self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 2); self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        x = self.fc2(self.relu(self.fc1(self.dropout(h_n[-1]))))
        return self.softmax(x)

model = DetectorLacy().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=0.000025)
criterion = nn.CrossEntropyLoss()

# Bucle de Entrenamiento
h_train, h_val = [], []
for epoch in range(epochs):
    model.train(); tr_loss = 0
    for x, l in train_loader:
        x, l = x.to(DEVICE), l.to(DEVICE)
        optimizer.zero_grad(); loss = criterion(model(x), l)
        loss.backward(); optimizer.step(); tr_loss += loss.item()
    
    model.eval(); va_loss = 0
    with torch.no_grad():
        for x, l in test_loader:
            x, l = x.to(DEVICE), l.to(DEVICE); va_loss += criterion(model(x), l).item()
    
    h_train.append(tr_loss/len(train_loader)); h_val.append(va_loss/len(test_loader))
    print(f"Epoch {epoch+1}/{epochs} | Loss: {h_train[-1]:.4f} | Val: {h_val[-1]:.4f}")

# Visualización de Loss
fig = go.Figure()
fig.add_trace(go.Scatter(y=h_train, name='Train', line=dict(color='cyan')))
fig.add_trace(go.Scatter(y=h_val, name='Val', line=dict(color='magenta')))
fig.update_layout(title="Historial de Entrenamiento", template="plotly_dark").show()

In [ ]:
def test_visual_deteccion_con_espectrograma(target=0, snr=30, index=0, batch_inf=512):
    model.eval()
    
    # 1. Obtener la muestra real
    tensor, _, _, _ = obtener_una_muestra(target=target, snr=snr, index=index)
    iq_raw = tensor.cpu().numpy()
    
    # 2. Pre-procesado Opción B (Magnitud + Decimación)
    # Usamos FACTOR_DECIMACION=70 para que 100 puntos = 0.5ms
    mag_comp = preprocesar_opcion_b(iq_raw, factor=FACTOR_DECIMACION)
    
    if mag_comp is None: 
        return print("⚠️ Error: La señal es demasiado corta para procesar.")
    
    # 3. Ventanas deslizantes para la LSTM
    seq_len = 100
    step = 20 
    ventanas, indices_t = [], []
    for i in range(0, len(mag_comp) - seq_len, step):
        ventanas.append(mag_comp[i:i+seq_len].reshape(-1, 1))
        indices_t.append(i * FACTOR_DECIMACION) 
    
    # 4. Inferencia
    with torch.no_grad():
        b = torch.FloatTensor(np.array(ventanas)).to(DEVICE)
        probs = model(b)[:, 1].cpu().numpy()
    
    # --- 5. CÁLCULO DEL ESPECTROGRAMA ---
    x_complex = iq_raw[0] + 1j * iq_raw[1]
    f_spec, t_spec, Pxx = signal.spectrogram(x_complex, fs=FREQ_MUESTREO, 
                                             nperseg=1024, noverlap=512, 
                                             return_onesided=False)
    
    f_spec = np.fft.fftshift(f_spec)
    Pxx = np.fft.fftshift(Pxx, axes=0)
    Pxx_db = 10 * np.log10(Pxx + 1e-15)
    t_spec_samples = t_spec * FREQ_MUESTREO
    
    # --- 6. VISUALIZACIÓN (3 FILAS) ---
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                        subplot_titles=("Espectrograma de Confirmación", 
                                        "Detección LSTM (Verde = Prob > 0.5)"))
    
    t_samples = np.arange(len(iq_raw[0]))
    mag_orig = np.sqrt(iq_raw[0]**2 + iq_raw[1]**2)
    # Fila 2: Espectrograma (SIN el error de 'layer')
    fig.add_trace(go.Heatmap(x=t_spec_samples, y=f_spec, z=Pxx_db, 
                             colorscale='Viridis', showscale=False), row=1, col=1)
    
    # Fila 3: Señal sombreada
    fig.add_trace(go.Scatter(x=t_samples, y=mag_orig, name="Detección", 
                             line=dict(color='white', width=1)), row=2, col=1)
    
    # Sombreado de detecciones (Aquí 'layer' SÍ es válido porque es un vrect)
    for i, p in enumerate(probs):
        if p > 0.5:
            x0_orig = indices_t[i]
            x1_orig = x0_orig + (seq_len * FACTOR_DECIMACION) 
            fig.add_vrect(x0=x0_orig, x1=x1_orig, fillcolor="green", opacity=0.3, 
                          layer="below", line_width=0, row=2, col=1)
    
    fig.update_layout(height=850, template="plotly_dark", showlegend=False,
                      title_text=f"Validación: {TARGET_MAP[target]} | SNR {snr}dB")
    fig.show()


In [ ]:
# --- EJECUTAMOS PRUEBA EN UN CASO DIFÍCIL (-10dB) ---
test_visual_deteccion_con_espectrograma(target=0, snr=0, index=13)